The electric field in a capacitor inspired by Joachim Schöberl

In [ ]:
from netgen.meshing import Mesh
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

In [ ]:
def CapacitorGeometry():

    air = MoveTo(0, 0).RectangleC(30, 30).Face()
    air.edges.name = "Outer"
    air.faces.name = "air"

    electrode_positive = MoveTo(0, 1).RectangleC(5, 0.5).Face()
    electrode_positive.edges.name = "electrode_positive"
    electrode_positive.faces.name = "electrode_positive"

    electrode_negative = MoveTo(0, -1).RectangleC(5, 0.5).Face()
    electrode_negative.edges.name = "electrode_negative"
    electrode_negative.faces.name = "electrode_negative"

    dielectric = MoveTo(0, 0).RectangleC(4, 1.5).Face()
    dielectric.faces.name = "dielectric"

    shape = Glue([air - dielectric, dielectric])
    shape = shape - electrode_positive - electrode_negative
    
    return shape


def CapacitorMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=h_max))

    return mesh


def CapacitorSolver(mesh, FE_order, epsr):

    fes = H1(mesh, order=FE_order, dirichlet="el.*")

    u = fes.TrialFunction()
    v = fes.TestFunction()

    solution_gf = GridFunction(fes)
    solution_gf.Interpolate(mesh.BoundaryCF({"electrode_positive":1, "electrode_negative":-1 }), mesh.Boundaries(".*"))

    a = BilinearForm(epsr*grad(u)*grad(v)*dx).Assemble()
    
    inv = a.mat.Inverse(freedofs=fes.FreeDofs())
    solution_gf.vec.data -= inv@a.mat * solution_gf.vec

    return solution_gf

In [ ]:
geo = CapacitorGeometry()
Draw(geo);

In [ ]:
h_max = 0.5
mesh = CapacitorMesh(geo, h_max)
Draw (mesh);

In [ ]:
epsr_air = 1.0
epsr_dielectric = 2.0

epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

Draw(epsr, mesh);

In [ ]:
FE_order = 1

phi_gf = CapacitorSolver(mesh, FE_order, epsr)

In [ ]:
Draw (phi_gf, deformation=True, scale=5);

In [ ]:
fes_flux = HDiv(mesh, order=FE_order-1)

E_gf = GridFunction(fes_flux)
D_gf = GridFunction(fes_flux)
E_gf.Set(-grad(phi_gf))
D_gf.Set(epsr*E_gf)

In [ ]:
Draw (E_gf, mesh, vectors= {"grid_size": 100});

In [ ]:
Draw (Norm(E_gf), mesh, deformation=True, min=0, max=2);

In [ ]:
Draw (D_gf, mesh, vectors= {"grid_size": 100});

In [ ]:
Draw (Norm(D_gf), mesh, deformation=True);

In [ ]:
N = 400
p = [(-10 + 0.05*i, -10 + 0.1*j, 0) for i in range(N) for j in range(N) ] 

fieldlines = E_gf._BuildFieldLines(mesh, p, num_fieldlines=300, length=0.3)

Draw(E_gf, mesh,  "X", draw_vol=True, draw_surf=True, objects=[fieldlines], \
     autoscale=True, min = 0, max = 2, settings={"Objects": {"Surface": False}});

In [ ]:
energy = 0.5 * Integrate(epsr*InnerProduct(E_gf, E_gf), mesh)
print(energy)